In [1]:
# Встановлення необхідних бібліотек для симуляції
try:
    import pandas, numpy, plotly, IPython
except ImportError:
    %pip install pandas numpy plotly ipython

<div align="center">
    <h2><b>Architecture Design Document (ADD)</b></h2>
    <h1>Обробка великих обсягів даних у системах (Big Data & Streaming)</h1>
    <h3>Муніципальна система «ctOSmartCity»</h3>
    <br>

**Метадані документа:**

| **Тип документа:** | High-Level Design (HLD) & Communication Strategy |
| :--- | :--- |
| **Статус:** | Draft / В розробці |
| **Підрядник:** | WatchDog Analytics (Blume Corporation) |
| **Версія архітектури:** | 1.0.0 |
| **Дата створення:** | Березень 2026 р. |

<br>

**Автор проєкту:**

| **Ім'я:** | Олег Гаценко |
| :--- | :--- |
| **Роль:** | Data Architect / Студент магістратури |
| **Організація:** | NeoVersity (Master's in AI & Machine Learning) |

</div>

---

### **Executive Summary (Резюме архітектури):**

Цей документ описує високорівневу архітектуру для платформи розумного міста **ctOSmartCity**. Система щосекунди агрегує події від тисяч IoT-сенсорів (світлофори, рівень CO₂, камери трафіку, паркінги). 

Оскільки система має одночасно забезпечувати **миттєве реагування** (виклик екстрених служб під час аварій) та **глибоку історичну аналітику** (місячні звіти для мерії), ми обрали класичний патерн обробки Big Data — **Лямбда-архітектуру (Lambda Architecture)**.

**Ключові архітектурні рішення:**
* **Ingestion Layer:** Асинхронний збір подій через `Apache Kafka` (єдине джерело правди та надійний буфер, що згладжує пікові навантаження).
* **Speed Layer (Гарячий шлях):** `Apache Flink` для Complex Event Processing (CEP) та виявлення аномалій у реальному часі.
* **Batch Layer (Холодний шлях):** Зберігання сирих даних у Data Lake (`HDFS` / `AWS S3`) та їх пакетна обробка через `Apache Spark`.
* **Serving Layer:** Швидка NoSQL база `Apache Cassandra` для відображення метрик на міських дашбордах та у мобільних застосунках містян.

### **Контекстна діаграма потоків (Level 0):**

```mermaid
graph LR
    %% Styling
    classDef source fill:#2b2b2b,stroke:#00ffcc,stroke-width:2px,color:#fff
    classDef stream fill:#1a1a1a,stroke:#ff3366,stroke-width:2px,color:#fff
    classDef batch fill:#1a1a1a,stroke:#3399ff,stroke-width:2px,color:#fff
    classDef sink fill:#2b2b2b,stroke:#ffcc00,stroke-width:2px,color:#fff

    IoT(("📡 IoT<br>Сенсори")):::source
    Kafka{{"📥 Ingestion<br>(Apache Kafka)"}}:::source
    
    Flink["⚡ Speed Layer<br>(Apache Flink)"]:::stream
    Spark["🧊 Batch Layer<br>(HDFS + Spark)"]:::batch
    
    DB[("📊 Serving<br>(Cassandra)")]:::sink
    Alerts[["🚨 Екстрені<br>Служби"]]:::sink

    IoT -- "Мільйони подій/сек" --> Kafka
    Kafka -- "Безперервний стрім" --> Flink
    Kafka -- "Скидання логів" --> Spark
    
    Flink -- "Миттєвий Alert (Exactly-Once)" --> Alerts
    Flink -- "Оновлення Real-time метрик" --> DB
    Spark -- "Важкі історичні агрегації" --> DB
```

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **1. Архітектура системи (Лямбда-архітектура)**

Для реалізації платформи **ctOSmartCity** обрано патерн **Lambda Architecture**, який чітко розмежовує потокову (Stream) та пакетну (Batch) обробку. Це гарантує, що важкі аналітичні запити (формування місячних звітів) ніколи не вплинуть на швидкість реакції системи на критичні події (аварії).

### 1.1. Основні компоненти та їх ролі:

- **Джерела подій (IoT Devices):** Тисячі міських сенсорів (камери трафіку, датчики CO₂, розумні паркінги). Дані передаються через легковісний протокол **MQTT** на IoT Gateway.
- **Транспорт (Ingestion Layer) — Apache Kafka:**
    -  *Роль:* Центральна "нервова система" міста, єдине джерело правди (Single Source of Truth) та розподілений буфер.
    - *Механіка:* Сирі події (у форматі JSON) записуються у відповідні топіки. Kafka згладжує будь-які пікові навантаження та дозволяє паралельно читати одні й ті самі дані різним підсистемам.
- **Потокова обробка (Speed Layer) — Apache Flink:**
    - *Роль:* Complex Event Processing (CEP) та аналіз у реальному часі.
    - *Механіка:* Безперервно читає стрім із Kafka, миттєво знаходить патерни (наприклад, фіксація ДТП + утворення затору) та відправляє Webhook-тригери у служби реагування.
- **Пакетна обробка та зберігання (Batch Layer):**
    - **HDFS (Data Lake):** Сховище для довгострокового зберігання "холодних" сирих даних (у стислому форматі Parquet) для майбутнього аналізу та прогнозування.
    - **Apache Spark:** Виконує важкі обчислення за розкладом (наприклад, щоночі). Агрегує терабайти історичних даних з HDFS для формування статистичних зведень для мерії.
- **Аналітика (Serving Layer) — Apache Cassandra:**
    - *Роль:* Швидка розподілена NoSQL база даних, оптимізована для читання. Flink оновлює тут live-метрики (поточний трафік), а Spark записує готові зведені звіти. Міські дашборди читають дані безпосередньо звідси.
- **Інтерфейс для служб реагування:**
    - *Роль:* REST API (Webhooks), які викликаються безпосередньо з Apache Flink для миттєвої передачі критичних інцидентів (виклик поліції, швидкої чи екологічної служби).

---

### 1.2. Механізми масштабування та відмовостійкості:

1. **Партиціонування (Partitioning):** Топіки в Kafka розбиті на партиції. Це дозволяє здійснювати горизонтальне масштабування — одночасно читати потік даних сотнями паралельних вузлів (TaskManagers) у Flink та Spark.
2. **Реплікація (Replication):** Kafka-топіки та блоки в HDFS мають фактор реплікації (зазвичай = 3). Якщо один фізичний сервер виходить з ладу, система миттєво перемикається на копії даних без їх втрати.
3. **Контрольні точки (Checkpointing):** `Apache Flink` періодично робить асинхронні знімки свого внутрішнього стану (State) та зберігає їх у HDFS. У разі падіння обчислювального вузла, система піднімає резервний контейнер і відновлює обробку з останнього чекпоінту, гарантуючи стійкість до збоїв.

Ми розділяємо потоки даних на два незалежні шляхи. Це гарантує, що важкі аналітичні запити (Spark) ніколи не заблокують критичний процес виявлення аварій (Flink).

**Детальна архітектурна схема системи ctOSmartCity (Lambda Architecture):**

```mermaid
flowchart TD
    subgraph IoT ["Джерела&nbsp;подій&nbsp;(IoT&nbsp;Devices)"]
        S1(["Камери<br/>Трафіку"])
        S2(["Сенсори<br/>Повітря CO2"])
        S3(["Розумні<br/>Світлофори"])
    end

    Gateway{{"IoT Gateway / MQTT"}}
    IoT -->|Стрім подій| Gateway

    Kafka[["Apache Kafka<br/>(Event Bus / Buffer)"]]
    Gateway -->|Raw JSON| Kafka

    subgraph SpeedLayer ["Speed Layer (Real-Time)"]
        Flink["Apache Flink<br/>(Потокова обробка)"]
    end

    subgraph BatchLayer ["Batch Layer (Cold Data)"]
        HDFS[("HDFS / AWS S3<br/>Data Lake")]
        Spark["Apache Spark<br/>(Пакетна обробка)"]
    end

    Kafka -->|Миттєве читання| Flink
    Kafka -.->|Скидання логів| HDFS
    HDFS -->|Щогодинні Job-и| Spark

    DB[("Apache Cassandra<br/>Serving DB")]

    Spark -->|Агреговані звіти| DB
    Flink -->|Оновлення метрик| DB

    Police{{"Екстрені Служби<br/>(API Webhooks)"}}
    Flink -->|Exactly-Once Alert| Police

    Dash(["Дашборди ctOSmartCity /<br/>Мобільний Застосунок"])
    DB -->|GraphQL / REST| Dash

    classDef external fill:#ffebee,stroke:#c62828,stroke-width:2px,color:#000;
    classDef stream fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef batch fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;

    class Police external;
    class Flink stream;
    class Spark,HDFS batch;
```

---

### 1.3. Обґрунтування архітектури: Оцінка навантаження (Capacity Planning):

Для підтвердження обраного технологічного стеку проведемо базові розрахунки (Back-of-the-envelope calculations) для середнього мегаполіса. 

**Вихідні дані (Assumptions):**
- Кількість IoT-сенсорів у місті: **100 000** пристроїв.
- Частота генерації подій: **1 подія/секунду** від кожного сенсора.
- Середній розмір одного повідомлення (JSON payload): **500 Bytes**.

**1. Пропускна здатність для Real-time обробки (Вимога 1):**
- **Throughput (RPS):** `100 000 events/sec`
- **Bandwidth (Вхідний трафік):** `100 000 * 500 B = 50 MB/sec`
- **Добовий трафік:** `50 MB/sec * 3600 sec * 24 h ≈ 4.3 TB/day`
> *Висновок:* Звичайні реляційні БД (PostgreSQL) не витримають безперервного потоку запису 100 тис. рядків на секунду. Саме тому на вході (Ingestion) стоїть **Apache Kafka**, здатна обробляти мільйони повідомлень за секунду завдяки послідовному запису на диск (Sequential I/O).

**2-3. Обсяг зберігання для довгострокової історії (Вимоги 2 та 3):**
- **Сирі дані за рік:** `4.3 TB/day * 365 days ≈ 1.5 PB/year`
- **З урахуванням реплікації (HDFS Replication Factor = 3):** `1.5 PB * 3 = 4.5 PB/year`
- **Оптимізація зберігання:** Оскільки ми використовуємо пакетну обробку (Batch Layer), сирі JSON-дані стискаються та конвертуються у колонковий формат **Apache Parquet** (алгоритм Snappy), що дає економію місця близько 70-80%
- **Реальний обсяг дисків:** `4.5 PB * 0.3 (30%) ≈ 1.35 PB/year`
> *Висновок:* Зберігати 1.35 Петабайта даних на рік у звичайній SQL-базі економічно неможливо. Тому для історичного аналізу використовується **HDFS** (дешеві HDD диски) та **Apache Spark**, який вміє розподілено читати Parquet-файли, не завантажуючи їх цілком в оперативну пам'ять.